In [14]:
import pubchempy as pcp
import pandas as pd
from scipy.io import loadmat
from keras.utils import to_categorical
from tensorflow.keras.layers import Input, Dense, Dropout, BatchNormalization
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import tensorflow as tf
import numpy as np

This model is a following the process at https://towardsdatascience.com/land-cover-classification-of-hyperspectral-imagery-using-deep-neural-networks-2e36d629a40e to learn how to read in .mat datasets, as well as formatting the pandas dataframe.

In [5]:
data = loadmat('data/Pavia.mat')['pavia']
labels = loadmat('data/Pavia_gt.mat')['pavia_gt']

In [6]:
data_df = pd.DataFrame(data.reshape(data.shape[0]*data.shape[1],-1))
data_df.columns = [f'band{i}' for i in range(1, data_df.shape[-1]+1)]
data_df['class'] = labels.ravel()

In [7]:
x = data_df[data_df['class']!=0].iloc[:,:-1].values
y = tf.keras.utils.to_categorical(data_df[data_df['class']!=0].iloc[:,-1].values, num_classes=np.unique(labels).shape[0],dtype='float32')

In [8]:
x_train, x_test, y_train, y_test = train_test_split(x,y,test_size=0.2,stratify=y)

In [11]:
from keras.models import Sequential
from keras.layers import Dense, Conv2D, Flatten

In [15]:
model = Sequential(name='Pavia_University')

model.add(Input(shape = x_train[0].shape, name = 'Input_Layer'))


model.add(Dense(units = 128, activation= 'relu', name = 'Layer1'))
model.add(Dense(units = 128, activation= 'relu', name = 'Layer2'))
model.add(Dense(units = 128, activation= 'relu', name = 'Layer3'))
model.add(Dense(units = 128, activation= 'relu', name = 'Layer4'))


model.add(Dense(units = 64, activation= 'relu', name = 'Layer5'))
model.add(Dense(units = 64, activation= 'relu', name = 'Layer6'))
model.add(Dense(units = 64, activation= 'relu', name = 'Layer7'))
model.add(Dense(units = 64, activation= 'relu', name = 'Layer8'))


model.add(Dense(units = 32, activation= 'relu', name = 'Layer9'))
model.add(Dense(units = 32, activation= 'relu', name = 'Layer10'))
model.add(Dense(units = 32, activation= 'relu', name = 'Layer11'))
model.add(Dense(units = 32, activation= 'relu', name = 'Layer12'))

model.add(Dense(units = y_train.shape[1], activation= 'softmax', name = 'Output_Layer'))

model.summary()

Model: "Pavia_University"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 Layer1 (Dense)              (None, 128)               13184     
                                                                 
 Layer2 (Dense)              (None, 128)               16512     
                                                                 
 Layer3 (Dense)              (None, 128)               16512     
                                                                 
 Layer4 (Dense)              (None, 128)               16512     
                                                                 
 Layer5 (Dense)              (None, 64)                8256      
                                                                 
 Layer6 (Dense)              (None, 64)                4160      
                                                                 
 Layer7 (Dense)              (None, 64)           

In [16]:
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3), loss='categorical_crossentropy',metrics=['accuracy'])

In [17]:
model.fit(x = x_train,y = y_train, validation_data = (x_test, y_test), epochs=5)

Epoch 1/5
3704/3704 [==============================] - 34s 9ms/step - loss: 0.2564 - accuracy: 0.9294 - val_loss: 0.1188 - val_accuracy: 0.9580
Epoch 2/5
3704/3704 [==============================] - 34s 9ms/step - loss: 0.1598 - accuracy: 0.9459 - val_loss: 0.1002 - val_accuracy: 0.9634
Epoch 3/5
3704/3704 [==============================] - 29s 8ms/step - loss: 0.1171 - accuracy: 0.9590 - val_loss: 0.1013 - val_accuracy: 0.9647
Epoch 4/5
3704/3704 [==============================] - 32s 9ms/step - loss: 0.1019 - accuracy: 0.9636 - val_loss: 0.1006 - val_accuracy: 0.9642
Epoch 5/5
3704/3704 [==============================] - 32s 9ms/step - loss: 0.0932 - accuracy: 0.9666 - val_loss: 0.0846 - val_accuracy: 0.9680


array([[1., 0., 0., ..., 0., 0., 0.],
       [1., 0., 0., ..., 0., 0., 0.],
       [1., 0., 0., ..., 0., 0., 0.],
       ...,
       [1., 0., 0., ..., 0., 0., 0.],
       [1., 0., 0., ..., 0., 0., 0.],
       [1., 0., 0., ..., 0., 0., 0.]], dtype=float32)